# Creating a Simple Agent with Tracing

In [2]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [3]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

In [5]:
placement_trainer_agent = Agent(
    name="Placement Trainer Agent",
    instructions="""
    You are a supportive placement trainer who prepares students for job interviews.
    Start by asking about the student's target role, experience level, and interview type.
    Then run a realistic mock interview, asking one question at a time and waiting for
    the student's answer before continuing. Include behavioral, role-specific, and
    introductory questions as appropriate.

    After each answer, give concise, constructive feedback: identify what worked,
    suggest one or two specific improvements, and offer an example of a stronger
    response when useful. Encourage structured answers such as STAR for behavioral
    questions. Do not invent details about the student's experience; ask follow-up
    questions when information is missing. Adapt the difficulty to the student and
    finish with a brief summary of strengths and practice priorities.
    """,
)

Let's execute the Agent:

In [6]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy and convenient. Key points:
    
    - Nutrients: potassium, vitamin C, vitamin B6, and dietary fiber; low in fat and protein.
    - Benefits: supports heart health, digestion, and steady energy from natural sugars.
    - Considerations: higher in carbs/sugars (ripeness matters; greener = more resistant starch), portions matter for blood sugar.
    - Special cases: kidney disease where potassium is restricted; watch added toppings (syrup, chocolate).
    
    Typical guidance: 1–2 bananas per day as part of a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [7]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are very healthy in moderation. Key points:

- Good sources of potassium (supports heart and fluid balance), vitamin B6, vitamin C, and dietary fiber.
- Contain natural sugars and carbs; ripe bananas are sweeter.
- Benefits: supports heart health, digestion, and steady energy.
- Watch portions if you’re monitoring sugar or total carbs (about 105 kcal per medium). One to two per day is reasonable for most people.
- Low allergen; peel protects most contaminants. wash before eating.

_Good Job!_